## Setup

In [1]:
import numpy as np
    
item_count = 16
param_path = "D:/data/base_cmr_parameters.json"

# mfc activations decide context_input; 
# there's also a normalization step
context_input = np.array([0., 0.89544394, 0., 0., 0., 0.,
                          0., 0., 0., 0., 0., 0.,
                          0., 0., 0., 0., 0., 0.])
context_input = context_input / np.sqrt(np.sum(np.square(context_input)))

print(context_input)

[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## Numba Implementation

In [2]:
from compmempy.models.context import numba_TemporalContext 
from compmempy.parameters import Parameters
parameters = Parameters(param_path).fixed

context_input = np.array(context_input)
items = np.eye(item_count)
numba_context = numba_TemporalContext(items, parameters)
numba_context.integrate(context_input, parameters['encoding_drift_rate'])

print(numba_context.state())

[0.5978168  0.80163276 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.        ]


In [3]:
%%timeit 
numba_context = numba_TemporalContext(np.eye(item_count), parameters)
numba_context.integrate(context_input, parameters['encoding_drift_rate'])

25.9 µs ± 1.12 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Jax Implementation

In [4]:
from jaxcmr.context import TemporalContext, integrate
from jax import numpy as jnp, jit
import json

with open(param_path) as f:
    parameters = json.load(f)['fixed']
context_input = jnp.array(context_input)

jax_context = TemporalContext.create(item_count)
jax_context = integrate(jax_context, context_input, parameters['encoding_drift_rate'])

print(jax_context.state)

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


[0.59781677 0.80163276 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.        ]


In [5]:
@jit
def timing_f():
    jax_context = TemporalContext.create(item_count)
    jax_context = integrate(jax_context, context_input, parameters['encoding_drift_rate'])
    
timing_f()

%timeit timing_f()

179 µs ± 7.08 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
